In [0]:
configs = {
  "fs.azure.account.auth.type": "OAuth",
  "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
  "fs.azure.account.oauth2.client.id": "4c73a0e8-727f-4978-a885-c24c55d9f5b3",
  "fs.azure.account.oauth2.client.secret": "secretid",
  "fs.azure.account.oauth2.client.endpoint": "https://login.microsoftonline.com/3c9a95b3-9291-4ed2-9677-13948a6e69d0/oauth2/token"
}
dbutils.fs.mount(
  source = "abfss://input@adlsdevvbsorce001.dfs.core.windows.net/",
  mount_point = "/mnt/source",
  extra_configs = configs)

Out[1]: True

In [0]:
%fs
ls '/mnt/source/snowflake'


path,name,size,modificationTime
dbfs:/mnt/source/snowflake/2025/,2025/,0,1756997234000


In [0]:
from datetime import datetime, timedelta

# yesterday’s file (assuming you process next day)
#process_date = (datetime.today() - timedelta(days=1)).strftime("%Y/%m/%d")
process_date = datetime.today().strftime("%Y/%m/%d") #--today's file

path = f"/mnt/source/snowflake/{process_date}/*.parquet"


raw_df = spark.read.format("parquet") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(path)

display(raw_df)

INDEX,ORDER_ID,ORDER_ITEM_ID,PRODUCT_ID,SELLER_ID,SHIPPING_LIMIT_DATE,PRICE,FREIGHT_VALUE
1,ORD1001,1,PROD001,SELLER01,2025-09-01T12:00:00.000+0000,199.99,10.50
2,ORD1002,1,PROD002,SELLER02,2025-09-05T15:30:00.000+0000,349.00,25.00
3,ORD1002,2,PROD003,SELLER03,2025-09-06T09:15:00.000+0000,120.75,12.25
4,ORD1003,1,PROD004,SELLER01,2025-09-10T18:00:00.000+0000,450.00,30.00
5,ORD1001,1,PROD001,SELLER01,2025-09-01T12:00:00.000+0000,201.99,78.50
6,ORD1002,2,PROD002,SELLER02,2025-09-05T15:30:00.000+0000,145.00,45.00
7,ORD1002,2,PROD003,SELLER03,2025-09-06T09:15:00.000+0000,220.75,41.25
8,ORD1003,1,PROD004,SELLER01,2025-09-10T18:00:00.000+0000,564.00,12.00


In [0]:
raw_df.write.format("delta").mode("overwrite").save("/mnt/curated/cust_order_delta")
spark.sql("CREATE TABLE IF NOT EXISTS cust_order_delta USING DELTA LOCATION '/mnt/curated/cust_order_delta'")

Out[6]: DataFrame[]

###SCD1

In [0]:
from pyspark.sql.functions import current_date, lit, row_number
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# ---- Step 1: Get latest file path and folder date ----
folders = dbutils.fs.ls("/mnt/source/snowflake/")
latest_year = max([f.name.replace('/', '') for f in folders])

folders = dbutils.fs.ls(f"/mnt/source/snowflake/{latest_year}/")
latest_month = max([f.name.replace('/', '') for f in folders])

folders = dbutils.fs.ls(f"/mnt/source/snowflake/{latest_year}/{latest_month}/")
latest_day = max([f.name.replace('/', '') for f in folders])

latest_path = f"/mnt/source/snowflake/{latest_year}/{latest_month}/{latest_day}/*.parquet"
print(f"Reading from: {latest_path}")

# Construct file_date from folder
file_date = f"{latest_year}-{latest_month}-{latest_day}"

# ---- Step 2: Read raw data and add columns ----
raw_df = spark.read.parquet(latest_path)

spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

new_df = (raw_df
          .withColumn("ingest_date", current_date())   # actual load date
          .withColumn("file_date", lit(file_date)))    # folder date

# ---- Step 2.1: Deduplicate (avoid merge errors) ----
window = Window.partitionBy("ORDER_ID", "ORDER_ITEM_ID").orderBy(new_df["ingest_date"].desc())
new_df = (new_df
          .withColumn("rn", row_number().over(window))
          .filter("rn = 1")   # keep only latest record per ORDER_ID + ORDER_ITEM_ID
          .drop("rn"))

# ---- Step 3: Merge into Delta (SCD1) ----
delta_path = "/mnt/curated/cust_order_delta"

if not DeltaTable.isDeltaTable(spark, delta_path):
    # Initial load
    (new_df.write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .save(delta_path))
else:
    # Merge (SCD1: overwrite on match, insert on new)
    deltaTable = DeltaTable.forPath(spark, delta_path)
    (deltaTable.alias("t")
     .merge(new_df.alias("s"),
            "t.ORDER_ID = s.ORDER_ID AND t.ORDER_ITEM_ID = s.ORDER_ITEM_ID")
     .whenMatchedUpdateAll()
     .whenNotMatchedInsertAll()
     .execute())

# ---- Step 4: Reload and check ----
updated_df = (spark.read
              .format("delta")
              .option("mergeSchema", "true")
              .load(delta_path))

print("Before Merge:")
raw_df.show(10, truncate=False)

print("After Merge:")
updated_df.show(10, truncate=False)


Reading from: /mnt/source/snowflake/2025/09/06/*.parquet
Before Merge:
+-----+--------+-------------+----------+---------+-------------------+------+-------------+
INDEX|ORDER_ID|ORDER_ITEM_ID|PRODUCT_ID|SELLER_ID|SHIPPING_LIMIT_DATE|PRICE |FREIGHT_VALUE|
+-----+--------+-------------+----------+---------+-------------------+------+-------------+
1 |ORD1001 |1 |PROD001 |SELLER01 |2025-09-01 12:00:00|199.99|10.50 |
2 |ORD1002 |1 |PROD002 |SELLER02 |2025-09-05 15:30:00|349.00|25.00 |
3 |ORD1002 |2 |PROD003 |SELLER03 |2025-09-06 09:15:00|120.75|12.25 |
4 |ORD1003 |1 |PROD004 |SELLER01 |2025-09-10 18:00:00|450.00|30.00 |
5 |ORD1001 |1 |PROD001 |SELLER01 |2025-09-01 12:00:00|201.99|78.50 |
6 |ORD1002 |2 |PROD002 |SELLER02 |2025-09-05 15:30:00|145.00|45.00 |
7 |ORD1002 |2 |PROD003 |SELLER03 |2025-09-06 09:15:00|220.75|41.25 |
8 |ORD1003 |1 |PROD004 |SELLER01 |2025-09-10 18:00:00|564.00|12.00 |
+-----+--------+-------------+----------+---------+-------------------+------+-------------+

After Merge:
+-----+--------+-------------+----------+---------+-------------------+------+-------------+-----------+----------+
INDEX|ORDER_ID|ORDER_ITEM_ID|PRODUCT_ID|SELLER_ID|SHIPPING_LIMIT_DATE|PRICE |FREIGHT_VALUE|ingest_date|file_date |
+-----+--------+-------------+----------+---------+-------------------+------+-------------+-----------+----------+
1 |ORD1001 |1 |PROD001 |SELLER01 |2025-09-01 12:00:00|199.99|10.50 |2025-09-06 |2025-09-06|
2 |ORD1002 |1 |PROD002 |SELLER02 |2025-09-05 15:30:00|349.00|25.00 |2025-09-06 |2025-09-06|
3 |ORD1002 |2 |PROD003 |SELLER03 |2025-09-06 09:15:00|120.75|12.25 |2025-09-06 |2025-09-06|
4 |ORD1003 |1 |PROD004 |SELLER01 |2025-09-10 18:00:00|450.00|30.00 |2025-09-06 |2025-09-06|
1 |ORD1001 |1 |PROD001 |SELLER01 |2025-09-01 12:00:00|199.99|10.50 |2025-09-06 |2025-09-06|
3 |ORD1002 |2 |PROD003 |SELLER03 |2025-09-06 09:15:00|120.75|12.25 |2025-09-06 |2025-09-06|
4 |ORD1003 |1 |PROD004 |SELLER01 |2025-09-10 18:00:00|450.00|30.00 |2025-09-06 |2025-09-06|
3 |ORD1002 |2 |PROD003 |SELLER03 |2025-09-06 09:15:00|120.75|12.25 |2025-09-06 |2025-09-06|
+-----+--------+-------------+----------+---------+-------------------+------+-------------+-----------+----------+